# EmbeddingGemma : conclusion

Dans le premier notebook, le français de référence venait d'un traducteur automatique, donc les scores étaient trop optimistes. Ici on refait les tests avec une vraie traduction humaine (corpus News Commentary, ou le jeu T/R du groupe si on a le fichier).

Trois questions :
1. Les scores d'EmbeddingGemma sont-ils fiables avec des références humaines ?
2. Est-il meilleur ou moins bon que COMETKiwi comme contrôleur de qualité ?
3. Peut-il aider Qwen3 à mieux traduire en choisissant les exemples du prompt ?

Pour la partie 3, Qwen traduit de trois façons : sans exemple, avec des exemples au hasard, et avec les exemples les plus proches de la phrase (choisis par EmbeddingGemma). La version au hasard sert à vérifier que c'est bien le choix des exemples qui aide, et pas juste le fait d'en avoir.

Avant de lancer : runtime GPU (T4), licence Gemma acceptée, secret HF_TOKEN dans Colab.

Attention, News Commentary c'est de la presse écrite, pas du journal télévisé parlé. Les valeurs absolues seront différentes du test final, ce qui compte c'est la comparaison entre méthodes sur les mêmes phrases.

In [ ]:
!pip install -q -U "sentence-transformers>=5.0" "transformers>=4.56" accelerate sentencepiece sacrebleu datasets scikit-learn scipy matplotlib openpyxl

In [ ]:
import os, sys, re, time, random, gc
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
IN_COLAB = "google.colab" in sys.modules
print("device :", device)

# paramètres
T_PATH      = None      # jeu T/R du groupe (colonnes ar, fr), None = corpus public News Commentary
BASE_PATH   = None      # base d'exemples du groupe (jeu D/P), None = corpus public
N_TEST      = 150       # phrases de test
N_BASE      = 5000      # taille de la réserve d'exemples (partie 3)
K_SHOTS     = 3         # exemples donnés à Qwen
MIN_WORDS, MAX_WORDS = 8, 45
EMB_MODEL   = "google/embeddinggemma-300m"
MT_SYSTEMS  = {"opus-mt-ar-fr (S0)": "Helsinki-NLP/opus-mt-ar-fr", "nllb-200-600M": "facebook/nllb-200-distilled-600M"}
LLM_NAME    = "Qwen/Qwen3-4B"           # Qwen3-1.7B si manque de mémoire
RUN_QWEN    = True                      # partie 3
RUN_COMETKIWI = False                   # partie 2, à lancer en dernier

In [ ]:
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
from huggingface_hub import login
if os.environ.get("HF_TOKEN"):
    login(token=os.environ["HF_TOKEN"])
else:
    login()

## Données de test
- Test : N_TEST phrases arabes avec leur traduction française humaine, pour noter les systèmes.
- Réserve d'exemples : N_BASE autres paires, sans aucune phrase en commun avec le test. EmbeddingGemma y cherche les exemples dans la partie 3.

In [ ]:
DIACRITICS = re.compile(r"[\u064B-\u0652\u0670]")
def clean_ar(t):
    t = str(t).replace("\u0640", "")
    t = DIACRITICS.sub("", t)
    t = re.sub(r"\s+([،\.؟\?!؛;:,])", r"\1", t)
    return re.sub(r"\s+", " ", t).strip()

def read_table(path):
    return pd.read_json(path, lines=True) if str(path).endswith(".jsonl") else pd.read_csv(path)

def public_pairs():
    from datasets import load_dataset
    ds = load_dataset("Helsinki-NLP/news_commentary", "ar-fr", split="train")
    rows = [(clean_ar(x["translation"]["ar"]), str(x["translation"]["fr"]).strip()) for x in ds]
    df = pd.DataFrame(rows, columns=["ar", "fr"])
    nw_ar, nw_fr = df.ar.str.split().str.len(), df.fr.str.split().str.len()
    ok = nw_ar.between(MIN_WORDS, MAX_WORDS) & (nw_fr / nw_ar).between(0.8, 2.5)
    return df[ok].drop_duplicates("ar").sample(frac=1, random_state=SEED).reset_index(drop=True)

pool = None
if T_PATH and os.path.exists(T_PATH):
    t = read_table(T_PATH)
    test = pd.DataFrame({"ar": t["ar"].map(clean_ar), "ref": t["fr"].astype(str)}).head(N_TEST)
    SOURCE_TEST = f"jeu du groupe ({T_PATH})"
else:
    pool = public_pairs()
    test = pool.iloc[:N_TEST].rename(columns={"fr": "ref"}).reset_index(drop=True)
    SOURCE_TEST = "News Commentary (public)"

if BASE_PATH and os.path.exists(BASE_PATH):
    b = read_table(BASE_PATH); base = pd.DataFrame({"ar": b["ar"].map(clean_ar), "fr": b["fr"].astype(str)}).head(N_BASE)
    SOURCE_BASE = f"base du groupe ({BASE_PATH})"
else:
    if pool is None: pool = public_pairs()
    start = N_TEST if SOURCE_TEST.startswith("News") else 0
    base = pool.iloc[start:start + N_BASE].reset_index(drop=True)
    SOURCE_BASE = "News Commentary (public)"

base = base[~base.ar.isin(set(test.ar))].reset_index(drop=True)      # pas de phrase commune entre test et réserve
print("Test :", len(test), "phrases,", SOURCE_TEST, "| Réserve d'exemples :", len(base), "paires,", SOURCE_BASE)
for i in range(2):
    print("AR :", test.ar[i][:130]); print("FR (humain) :", test.ref[i][:130]); print("-" * 30)

In [ ]:
from sentence_transformers import SentenceTransformer
from sacrebleu.metrics import CHRF, BLEU
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score

model = SentenceTransformer(EMB_MODEL, device=device)
S_PROMPT = "task: sentence similarity | query: "

def embed(m, texts, prompt="", bs=16):
    return m.encode(list(texts), prompt=prompt or None, batch_size=bs, convert_to_numpy=True,
                    normalize_embeddings=True, show_progress_bar=False)

chrfpp, bleu = CHRF(word_order=2), BLEU()
def seg_chrf(hyps, refs):
    return np.array([chrfpp.sentence_score(h, [r]).score for h, r in zip(hyps, refs)])
def corpus_scores(hyps, refs):
    return round(chrfpp.corpus_score(list(hyps), [list(refs)]).score, 1), round(bleu.corpus_score(list(hyps), [list(refs)]).score, 1)

def paired_bootstrap(a, b, n=2000, seed=0):
    # a et b : scores par phrase, renvoie le gain moyen de a sur b et l'intervalle à 95 %
    rng = np.random.default_rng(seed); d = np.asarray(a) - np.asarray(b); m = len(d)
    means = [d[rng.integers(0, m, m)].mean() for _ in range(n)]
    return round(float(d.mean()), 2), (round(float(np.percentile(means, 2.5)), 2), round(float(np.percentile(means, 97.5)), 2))
print("modèle d'embedding prêt")

## Partie 1 : scores réels
Opus-MT (le S0 du groupe) et NLLB traduisent les phrases de test, notées avec chrF++ et BLEU contre la traduction humaine.

Ensuite on regarde si la note d'EmbeddingGemma (cosinus entre l'arabe et la traduction, sans la référence) suit chrF++ :
- corrélation de Spearman
- AUC entre les 25 % meilleures et les 25 % pires traductions selon chrF++ (probabilité que le contrôleur note la bonne plus haut que la mauvaise, 0,5 = hasard)
- filtre des 20 % : on retire les 20 % les plus mal notés par le contrôleur et on regarde le chrF++ moyen du reste, à comparer avec un filtre idéal qui retire les vrais pires

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

def translate(path, texts, bs=16, num_beams=4, max_new_tokens=256):
    is_nllb = "nllb" in path
    tok = AutoTokenizer.from_pretrained(path, **({"src_lang": "arb_Arab"} if is_nllb else {}))
    mt = AutoModelForSeq2SeqLM.from_pretrained(path).to(device).eval()
    kw = dict(num_beams=num_beams, max_new_tokens=max_new_tokens)
    if is_nllb: kw["forced_bos_token_id"] = tok.convert_tokens_to_ids("fra_Latn")
    texts = list(texts); order = np.argsort([len(t) for t in texts])[::-1]; out = [None] * len(texts)
    for i in tqdm(range(0, len(texts), bs), desc=path.split("/")[-1]):
        idx = order[i:i + bs]
        enc = tok([texts[j] for j in idx], return_tensors="pt", padding=True, truncation=True, max_length=256).to(device)
        with torch.no_grad():
            gen = mt.generate(**enc, **kw)
        for j, d in zip(idx, tok.batch_decode(gen, skip_special_tokens=True)):
            out[j] = d
    del mt; gc.collect(); torch.cuda.empty_cache()
    return out

HYP = {name: translate(path, test.ar) for name, path in MT_SYSTEMS.items()}
rows = []
for name, h in HYP.items():
    c, b = corpus_scores(h, test.ref); rows.append({"système": name, "chrF++": c, "BLEU": b})
res_scores = pd.DataFrame(rows); display(res_scores)

In [ ]:
E_ar = embed(model, test.ar, S_PROMPT)
COS, CHRF_SEG = {}, {}
rows = []
for name, h in HYP.items():
    COS[name] = (E_ar * embed(model, h, S_PROMPT)).sum(1)
    CHRF_SEG[name] = seg_chrf(h, test.ref)

def judge(cos, chrf, label):
    rho, _ = spearmanr(cos, chrf)
    lo, hi = np.quantile(chrf, 0.25), np.quantile(chrf, 0.75)
    bad, good = cos[chrf <= lo], cos[chrf >= hi]
    auc = roc_auc_score(np.r_[np.ones(len(good)), np.zeros(len(bad))], np.r_[good, bad])
    reject = cos <= np.quantile(cos, 0.20); oracle = chrf <= np.quantile(chrf, 0.20)
    return {"système": label, "Spearman(cos, chrF++)": round(float(rho), 3), "AUC bons/mauvais": round(float(auc), 3),
            "chrF++ moyen (tout)": round(float(chrf.mean()), 1),
            "après filtre EmbeddingGemma": round(float(chrf[~reject].mean()), 1),
            "après filtre idéal": round(float(chrf[~oracle].mean()), 1)}

for name in HYP: rows.append(judge(COS[name], CHRF_SEG[name], name))
rows.append(judge(np.concatenate(list(COS.values())), np.concatenate(list(CHRF_SEG.values())), "les deux ensemble"))
res_qe_real = pd.DataFrame(rows); display(res_qe_real)

allc, allf = np.concatenate(list(COS.values())), np.concatenate(list(CHRF_SEG.values()))
plt.figure(figsize=(5, 4)); plt.scatter(allc, allf, s=10, alpha=0.5)
plt.xlabel("note d'EmbeddingGemma (cos arabe/français)"); plt.ylabel("vraie qualité (chrF++)")
plt.title("La note suit-elle la vraie qualité ?"); plt.tight_layout(); plt.show()

## Partie 2 : EmbeddingGemma contre COMETKiwi
COMETKiwi est le contrôleur utilisé par le groupe (filtre D4-7). On note les mêmes traductions avec les deux et on compare la corrélation avec chrF++, le recouvrement des 20 % rejetés et le temps de calcul.

À lancer en dernier avec RUN_COMETKIWI = True, parce que unbabel-comet change des versions de bibliothèques. Il faut aussi avoir accepté la licence de Unbabel/wmt22-cometkiwi-da sur Hugging Face. Si ça casse, passer par le notebook cometkiwi_comparaison.

In [ ]:
res_kiwi = pd.DataFrame()
if RUN_COMETKIWI:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "unbabel-comet"], check=False)
    from comet import download_model, load_from_checkpoint
    kiwi = load_from_checkpoint(download_model("Unbabel/wmt22-cometkiwi-da"))
    rows = []
    for name, h in HYP.items():
        t0 = time.perf_counter()
        out = kiwi.predict([{"src": a, "mt": m} for a, m in zip(test.ar, h)], batch_size=16, gpus=1 if device == "cuda" else 0)
        t_kiwi = time.perf_counter() - t0
        k = np.array(out.scores)
        t0 = time.perf_counter(); e = (E_ar * embed(model, h, S_PROMPT)).sum(1); t_emb = time.perf_counter() - t0
        chrf = CHRF_SEG[name]
        wk, we = set(np.argsort(k)[:int(0.2 * len(k))]), set(np.argsort(e)[:int(0.2 * len(e))])
        keep_k = np.ones(len(k), bool); keep_k[list(wk)] = False
        keep_e = np.ones(len(e), bool); keep_e[list(we)] = False
        rows.append({"système": name,
                     "Spearman(COMETKiwi, chrF++)": round(float(spearmanr(k, chrf)[0]), 3),
                     "Spearman(EmbeddingGemma, chrF++)": round(float(spearmanr(e, chrf)[0]), 3),
                     "Spearman(COMETKiwi, EmbeddingGemma)": round(float(spearmanr(k, e)[0]), 3),
                     "recouvrement des 20 % rejetés": round(len(wk & we) / len(wk), 3),
                     "chrF++ après filtre COMETKiwi": round(float(chrf[keep_k].mean()), 1),
                     "chrF++ après filtre EmbeddingGemma": round(float(chrf[keep_e].mean()), 1),
                     "temps COMETKiwi (s)": round(t_kiwi, 1), "temps EmbeddingGemma (s)": round(t_emb, 1)})
    res_kiwi = pd.DataFrame(rows); display(res_kiwi)
else:
    print("COMETKiwi désactivé (RUN_COMETKIWI = False).")

## Partie 3 : EmbeddingGemma aide Qwen
Qwen traduit chaque phrase de test de trois façons :
1. sans exemple
2. avec K_SHOTS exemples tirés au hasard dans la réserve
3. avec les K_SHOTS exemples dont l'arabe est le plus proche, choisis par EmbeddingGemma

Le prompt système demande une traduction de sous-titres de journal télévisé. Pour savoir si un gain est réel, on calcule l'intervalle à 95 % de la différence (bootstrap apparié) : si 0 n'est pas dedans, le gain est fiable.

Limites : Qwen n'est pas affiné (pas de LoRA) alors que le S2 du groupe l'est, et la réserve et le test viennent du même corpus, ce qui avantage la recherche d'exemples.

In [ ]:
res_qwen, qwen_out = pd.DataFrame(), {}
if RUN_QWEN:
    from transformers import AutoModelForCausalLM
    llm_tok = AutoTokenizer.from_pretrained(LLM_NAME, padding_side="left")
    use_bf16 = device == "cuda" and torch.cuda.is_bf16_supported()
    llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype=torch.bfloat16 if use_bf16 else torch.float16).to(device).eval()

    SYSTEM = ("Tu es un traducteur professionnel. Traduis le texte arabe en français, pour des sous-titres de journal télévisé : "
              "phrase complète, naturelle, fidèle au sens, sans rien ajouter. Réponds uniquement par la traduction.")

    def make_prompt(ar, shots=()):
        msgs = [{"role": "system", "content": SYSTEM}]
        for a, f in shots:
            msgs += [{"role": "user", "content": a}, {"role": "assistant", "content": f}]
        msgs.append({"role": "user", "content": ar})
        return llm_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)

    def clean_llm(t):
        t = re.sub(r"<think>.*?</think>", "", t, flags=re.S)
        return t.strip()

    def llm_generate(prompts, bs=8, max_new_tokens=256):
        order = np.argsort([len(p) for p in prompts])[::-1]; out = [None] * len(prompts)
        for i in tqdm(range(0, len(prompts), bs), desc="Qwen"):
            idx = order[i:i + bs]
            enc = llm_tok([prompts[j] for j in idx], return_tensors="pt", padding=True).to(device)
            with torch.no_grad():
                gen = llm.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False)
            texts = llm_tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
            for j, t in zip(idx, texts): out[j] = clean_llm(t)
        return out

In [ ]:
if RUN_QWEN:
    B_emb = embed(model, base.ar, S_PROMPT); T_emb = embed(model, test.ar, S_PROMPT)
    S = T_emb @ B_emb.T
    top = np.argsort(-S, axis=1)[:, :K_SHOTS]
    rng = np.random.default_rng(SEED)
    shots = {
        "sans exemple": [()] * len(test),
        f"{K_SHOTS} exemples au hasard": [tuple((base.ar[j], base.fr[j]) for j in rng.choice(len(base), K_SHOTS, replace=False)) for _ in range(len(test))],
        f"{K_SHOTS} exemples choisis (EmbeddingGemma)": [tuple((base.ar[j], base.fr[j]) for j in row[::-1]) for row in top],   # le plus proche en dernier
    }
    print("Similarité moyenne du meilleur exemple retrouvé :", round(float(S.max(1).mean()), 3), "| au hasard :", round(float(S.mean()), 3))

    for name, sh in shots.items():
        qwen_out[name] = llm_generate([make_prompt(a, s) for a, s in zip(test.ar, sh)])

    rows = []
    for name, h in qwen_out.items():
        c, b = corpus_scores(h, test.ref); rows.append({"méthode": f"Qwen {name}", "chrF++": c, "BLEU": b})
    for name, h in HYP.items():
        c, b = corpus_scores(h, test.ref); rows.append({"méthode": f"{name} (référence)", "chrF++": c, "BLEU": b})
    res_qwen = pd.DataFrame(rows); display(res_qwen)

    names = list(qwen_out)
    seg = {n: seg_chrf(qwen_out[n], test.ref) for n in names}
    comp = [(names[2], names[0]), (names[2], names[1]), (names[1], names[0])]
    rows = []
    for a, b in comp:
        gain, (lo, hi) = paired_bootstrap(seg[a], seg[b])
        rows.append({"comparaison": f"{a}  contre  {b}", "gain chrF++ moyen": gain, "intervalle 95 %": f"[{lo} ; {hi}]", "gain fiable ?": "oui" if lo > 0 else ("non (peut-être pire)" if hi < 0 else "non (indécis)")})
    res_qwen_gain = pd.DataFrame(rows); display(res_qwen_gain)

    for i in [0, 1, 2]:
        print("AR :", test.ar[i][:140]); print("Humain :", test.ref[i][:140])
        for n in names: print(f"{n[:28]:>28} :", qwen_out[n][i][:140])
        print("-" * 40)
else:
    res_qwen_gain = pd.DataFrame(); print("Partie 3 désactivée (RUN_QWEN = False).")

## Sauvegarde

In [ ]:
with pd.ExcelWriter("resultats_conclusion_embeddinggemma.xlsx") as w:
    for name, tab in [("1_scores_MT", res_scores), ("1_QE_vraie_qualite", res_qe_real), ("2_COMETKiwi", res_kiwi),
                      ("3_Qwen_scores", res_qwen), ("3_Qwen_gains", res_qwen_gain)]:
        if len(tab): tab.to_excel(w, sheet_name=name, index=False)
    out = test.copy()
    for n, h in HYP.items(): out[f"hyp::{n}"] = h; out[f"cos::{n}"] = COS[n]; out[f"chrF::{n}"] = CHRF_SEG[n]
    for n, h in qwen_out.items(): out[f"qwen::{n}"] = h
    out.to_excel(w, sheet_name="donnees", index=False)
print("Écrit : resultats_conclusion_embeddinggemma.xlsx")
if IN_COLAB:
    from google.colab import files
    files.download("resultats_conclusion_embeddinggemma.xlsx")

## Lecture des résultats
- Partie 1 : si Spearman est autour de 0,5 ou plus et l'AUC au-dessus de 0,75, EmbeddingGemma suit bien la qualité. Si le filtre EmbeddingGemma est proche du filtre idéal il est utile, s'il reste proche de « tout » (hasard) il ne sert à rien.
- Partie 2 : s'il est proche de COMETKiwi en corrélation et plus rapide, il peut remplacer COMETKiwi au filtre D4-7. S'il est nettement derrière, on garde COMETKiwi.
- Partie 3 : l'aide d'EmbeddingGemma est prouvée seulement si le gain contre « au hasard » est fiable. Si les exemples choisis ne battent pas les exemples au hasard, ce n'est pas la recherche qui aide.
- Refaire le notebook avec le jeu T/R (T_PATH) et la base D/P (BASE_PATH) avant la décision finale.